# Классификация паттернов поведения: TimesFM → Laya (клон Jev)

**Задача:** по окну потребления мощности (P_RMS, мВт) умной розетки определить **паттерн поведения** (класс бытового устройства), закодированный в имени файла.

**Что в этом ноутбуке:**
1. Загрузка данных из **Google Drive** (как в исходном `TimesFM3_DeepSeek.ipynb`) — папка `SmartPlug`, окна по 90 отсчётов, метка из имени `plug_dump_..._<LABEL>_<DURATION>.csv`.
2. Классификация через **Laya** (`convaiinnovations/laya`) — открытая (Apache-2.0) Jev-совместимая модель: типизированные вопросы `choice`/`score`/`noul` + калиброванные вероятности, один forward pass, без генерации текста.
3. Сравнение с бейзлайном: Random Forest на тех же статистических признаках (TimesFM-вариант остаётся в исходном ноутбуке).
4. **Дообучение Laya** на наших данных: замена финальных слоёв на классификационную голову (число классов из данных) и supervised-обучение на train-сплите.
5. **Франкенштейн**: энкодер **Google TimesFM 3.0** + классификационная голова **Laya** (переходник 1280→1024). Все подходы сравниваются на одном и том же тесте.
6. **Улучшения Франкенштейна** (5.6–5.7): linear-probe «потолка», маскированный pool, z-нормализация, bottleneck-адаптер, валидация и macro-F1.

> Работает и в **Google Colab**, и в **VS Code** (Jupyter). В Colab данные монтируются из Google Drive, локально — из папки `SmartPlug` (переменная `SMARTPLUG_DIR` или `~/SmartPlug`).

⚠️ Честные ограничения Laya (из model card): базовые чекпойнты zero-shot на «голых» рядах могут быть близки к случайному угадыванию — наибольший выигрыш даёт дообучение на ваших данных. Практический эффект оценивайте по ячейке сравнения с бейзлайном.

## 0. Окружение и данные

In [ ]:
# 0.1 Установка Laya (клон Jev).
# Первый запуск скачает чекпойнт с Hugging Face (~800 МБ для english-версии).
%pip install -q laya
print("laya установлена")

In [ ]:
# 0.2 Импорты и общие настройки
import os, sys, re, json, warnings
from collections import Counter

os.environ.setdefault("USE_TF", "0")  # чтобы transformers не схватывал TF-рантайм при импорте

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier

from laya import Router

warnings.filterwarnings("ignore")
np.random.seed(42)

IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    pass

print("Среда:", "Google Colab" if IN_COLAB else "Локальная (VS Code)")
print("Python:", sys.version.split()[0])

# Устройство для Laya
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Устройство:", DEVICE, "| CUDA:", torch.cuda.is_available())
except ImportError:
    DEVICE = "cpu"
    print("Устройство: cpu (torch не найден)")

In [ ]:
# 0.3 Папка с данными
#   Colab  -> монтируем Google Drive и берём /content/drive/MyDrive/SmartPlug
#   VS Code-> переменная окружения SMARTPLUG_DIR либо ~/SmartPlug
DATA_DIR = os.environ.get("SMARTPLUG_DIR", "").strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = DATA_DIR or "/content/drive/MyDrive/SmartPlug"
else:
    DATA_DIR = DATA_DIR or os.path.expanduser("~/SmartPlug")

print("Папка данных:", DATA_DIR)
print("Существует:", os.path.isdir(DATA_DIR))
if not os.path.isdir(DATA_DIR):
    raise SystemExit(
        "Папка с данными не найдена. "
        "В Colab положите датасет в MyDrive/SmartPlug; "
        "локально — укажите SMARTPLUG_DIR или создайте ~/SmartPlug."
    )

## 1. Загрузка данных (как в исходном ноутбуке)

In [ ]:
# 1.1 Параметры разбиения
CHUNK_LENGTH = 90                  # длина одного окна (в отсчётах)
FILTER_OUT = ("IdleCharge", "MixedBrowsing", "Browsing", "VKAudio")  # классы, которые пропускаем
COLUMN = " P_RMS, mW"              # колонка мощности (мощность потребления)

filelist = sorted(f for f in os.listdir(DATA_DIR) if f.lower().endswith(".csv"))
print("Найдено файлов:", len(filelist))
if not filelist:
    raise SystemExit("В папке данных нет .csv-файлов.")

In [ ]:
# 1.2 Парсер имени файла: plug_dump_YYYY_MM_DD_hh_mm_ss_<LABEL>_<DURATION>.csv
def get_label_duration(filename: str):
    duration = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_\D+?_(\d+)\.csv", filename)
    label = re.findall(r"plug_dump_\d{4}_\d{2}_\d{2}_\d{2}_\d{2}_\d{2}_(\D+?)_\d+\.csv", filename)
    if not duration or not label:
        return None, None
    return label[0], duration[0]

# Проверка на примере
sample = filelist[0]
print("Пример имени:", sample)
print("Метка:", get_label_duration(sample))

In [ ]:
# 1.3 Чтение файлов и сбор окон длиной CHUNK_LENGTH
#     (копия логики исходного ноутбука; ищем колонку мощности по strip-имени, чтобы
#      не зависеть от лидирующего пробела в заголовке)
Labels, ndata, skipped = [], None, 0

for filename in tqdm(filelist, desc="Обработка файлов"):
    label, _ = get_label_duration(filename)
    if label is None:
        skipped += 1
        continue
    if any(tag in label for tag in FILTER_OUT):
        continue
    try:
        df = pd.read_csv(os.path.join(DATA_DIR, filename), delimiter=";")
        col = next((c for c in df.columns if c.strip() == COLUMN.strip()), None)
        if col is None:
            skipped += 1
            continue
        series = df[col].to_numpy(dtype=np.float64)
    except Exception:
        skipped += 1
        continue

    num_chunks = len(series) // CHUNK_LENGTH
    for i in range(num_chunks):
        chunk = series[i * CHUNK_LENGTH:(i + 1) * CHUNK_LENGTH]
        ndata = chunk[None, :] if ndata is None else np.vstack([ndata, chunk])
        Labels.append(label)

ndata = np.asarray(ndata, dtype=np.float32) if ndata is not None else np.empty((0, CHUNK_LENGTH))
print("Пропущено файлов:", skipped)
print("Всего фрагментов:", len(Labels), "| Форма массива:", ndata.shape)

print("Метки и их частоты:")
for u, c in Counter(Labels).items():
    print(f"  {u}: {c}")
if len(Labels) < 20:
    raise SystemExit("Слишком мало фрагментов — проверьте папку/фильтры.")

In [ ]:
# 1.4 Кодирование меток и сплит train/test (стратификация), как в исходнике
le = LabelEncoder()
y_encoded = le.fit_transform(Labels)
classes = list(le.classes_)
print("Классов:", len(classes), "| Классы:", ", ".join(classes))

X_train, X_test, y_train, y_test = train_test_split(
    ndata, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
test_labels = le.inverse_transform(y_test)
print("Train:", X_train.shape, "| Test:", X_test.shape)

## 2. Laya: классификация паттернов поведения

> **Почему train/test?** Laya более не обучается — ячейки 2.x используют модель «как есть» (zero-shot). Разбиение data нужна для **единой, честной метрики на тесте**: one сразу может сравнивать Laya zero-shot с Random Forest и (раздел 4) с дообученной головой на одном и том же тесте. Обучающая часть на нулевом прогоне не используется — но она пригодится для дообучения в разделе 4.

In [ ]:
# 2.1 Загрузка Router (маршрутизатор по языкам/чекпойнтам Laya).
# Первый predict скачает нужный чекпойнт (english ~800 МБ); preload=True держит все сразу.
router = Router(device=DEVICE)
print("Router готов. Устройство:", DEVICE)

In [ ]:
# 2.2 Числовой чанк -> "state" для Laya (компактное числовое описание + статистики),
#     и типизированные вопросы: паттерн поведения как choice по фактическим классам.
def chunk_to_state(chunk: np.ndarray, id_: int, *, n_points: int = 24) -> dict:
    chunk = np.asarray(chunk, dtype=np.float64)
    stats = {
        "mean_mw": float(np.mean(chunk)),
        "std_mw": float(np.std(chunk)),
        "min_mw": float(np.min(chunk)),
        "max_mw": float(np.max(chunk)),
        "p25_mw": float(np.percentile(chunk, 25)),
        "p50_mw": float(np.percentile(chunk, 50)),
        "p75_mw": float(np.percentile(chunk, 75)),
        "range_mw": float(np.ptp(chunk)),
        "energy_g": float(np.sum(chunk ** 2) / 1e6),
        "mean_abs_diff": float(np.mean(np.abs(np.diff(chunk)))),
        "trend_mw_per_pt": float(np.polyfit(np.arange(len(chunk)), chunk, 1)[0]),
    }
    idx = np.linspace(0, len(chunk) - 1, n_points).astype(int)
    return {
        "chunk_id": id_,
        "desc": "90-sample power-consumption window from a smart plug (P_RMS in mW)",
        "sampled_series_mw": [round(float(v), 1) for v in chunk[idx]],
        "stats": stats,
    }


def build_questions(classes, lang: str = "en") -> dict:
    if lang == "ru":
        instr = ("Какой паттерн поведения (класс бытового устройства) соответствует "
                 "этому окну потребления мощности? Оцените по статистикам и кривой мощности.")
    else:
        instr = ("Which behaviour pattern (home-appliance class) matches this "
                 "power-consumption window? Judge by the statistics and the power curve.")
    criteria = {c: f"device behaviour pattern {c}" for c in classes}
    return {"pattern": {"type": "choice", "instructions": instr, "criteria": criteria}}


QUESTIONS = build_questions(classes, lang="ru")
print(json.dumps(QUESTIONS, ensure_ascii=False, indent=2))

In [ ]:
# 2.3 Пробный прогон на нескольких чанках из теста
import numpy as np
demo_idx = np.random.RandomState(1).choice(len(X_test), size=min(5, len(X_test)), replace=False)
for i in demo_idx:
    state = chunk_to_state(X_test[i], id_=int(i))
    res = router.predict(state, QUESTIONS)
    ans = res["answers"]["pattern"]
    pred, conf = ans["choice"], float(ans["probabilities"][ans["choice"]])
    gt = test_labels[int(i)]
    ok = "OK" if pred == gt else "X"
    print(f"[{ok}] true={gt:<12s} pred={pred:<12s} conf={conf:.3f} routed={res['routing']['model']}")

In [ ]:
# 2.4 Массовая классификация тестовой выборки через Laya
MAX_N = 300          # ограничение размера для скорости; 0 = вся выборка
n = len(X_test) if MAX_N == 0 else min(MAX_N, len(X_test))
print(f"Прогоняем {n} чанков...")

preds, confs, routed = [], [], []
for i in tqdm(range(n), desc="Laya predict"):
    state = chunk_to_state(X_test[i], id_=int(i))
    res = router.predict(state, QUESTIONS)
    ans = res["answers"]["pattern"]
    preds.append(ans["choice"])
    confs.append(float(ans["probabilities"][ans["choice"]]))
    routed.append(res["routing"]["model"])

y_true_sub = le.inverse_transform(y_test[:n])
acc = accuracy_score(y_true_sub, preds)
print("=== Laya (клон Jev) ===")
print(f"Accuracy: {acc:.4f}")
print(f"Средняя уверенность: {np.mean(confs):.4f}")
print("Маршрутизация (использованные чекпойнты):", dict(Counter(routed)))

In [ ]:
# 2.5 Отчёт о качестве: классификация / матрица ошибок / точность по классам
print(classification_report(y_true_sub, preds))
cm = confusion_matrix(y_true_sub, preds)
plt.figure(figsize=(max(6, len(classes)), max(6, len(classes))))
plt.imshow(cm, cmap="Blues")
plt.colorbar()
plt.xticks(range(len(classes)), classes, rotation=45, ha="right")
plt.yticks(range(len(classes)), classes)
plt.xlabel("Предсказано"); plt.ylabel("Истина")
for i in range(len(classes)):
    for j in range(len(classes)):
        plt.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.title("Laya: confusion matrix")
plt.show()

## 3. Сравнение с бейзлайном

In [ ]:
# 3.1 Бейзлайн: Random Forest на тех же статистических признаках (без TimesFM).
#     Признаки те же, что уходят в "state" для Laya, — честное сравнение на входе.
STAT_KEYS = ["mean_mw", "std_mw", "min_mw", "max_mw", "p25_mw", "p50_mw", "p75_mw",
             "range_mw", "energy_g", "mean_abs_diff", "trend_mw_per_pt"]

def stats_vector(chunk: np.ndarray) -> np.ndarray:
    s = chunk_to_state(chunk, 0)["stats"]
    return np.array([s[k] for k in STAT_KEYS], dtype=np.float64)

X_feat = np.vstack([stats_vector(x) for x in tqdm(ndata, desc="Стат-признаки")])
print("Признаковое пространство:", X_feat.shape)

Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(
    X_feat, y_encoded, test_size=0.3, random_state=42, stratify=y_encoded
)
scaler = StandardScaler().fit(Xf_tr)
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(scaler.transform(Xf_tr), yf_tr)
rf_acc = accuracy_score(yf_te, rf.predict(scaler.transform(Xf_te)))
print(f"RandomForest (stats): {rf_acc:.4f}")

try:
    laya_acc = accuracy_score(y_true_sub, preds)
    print(f"Laya                : {laya_acc:.4f}")
    delta = laya_acc - rf_acc
    print(f"Разница Laya - RF   : {delta:+.4f}")
except NameError:
    print("Сначала выполните ячейку 2.4 (Laya predict).")

In [ ]:
# 3.2 (опционально) Отбор по уверенности: accuracy по покрытию
#     Показывает, насколько калиброван Laya: отсекая низкие confidence, растёт ли точность.
if len(set(confs)) > 1:
    thresholds = [0.5, 0.6, 0.7, 0.8, 0.9]
    print("threshold | accuracy | coverage")
    for t in thresholds:
        mask = np.array(confs) >= t
        if mask.sum() == 0:
            continue
        acc_t = accuracy_score(np.array(y_true_sub)[mask], np.array(preds)[mask])
        print(f"{t:9.2f} | {acc_t:.4f} | {mask.mean():.3f}")

## 4. Дообучение Laya (замена финальных классификационных слоёв)

В разделах 2–3 модель работала «как есть» (zero-shot). Теперь пользуемся тем, что у нас есть **размеченный датасет** с одинаковым train/test сплитом: дообучим классификатор поверх энкодера Laya.

**Идея (как ты предложил):** влезаем в структуру нейросети и меняем последний слой — вместо head со `scorer` ставим **классификационную голову** на `mean-pooled` представление `last_hidden_state` энкодера, с числом выходов `NUM_CLASSES = len(classes)` (определено в 1.4). Обучаем supervised: Cross-Entropy на train, оцениваем на том же test (ячейки 2.4/3.1).

Варианты объёма дообучения:
- `FREEZE_ENCODER = True` — обучается только новая голова (быстро, малый риск переобучения на скромном датасете);
- `FREEZE_ENCODER = False` — дообучается и энкодер с малым LR `2e-5` (нужна GPU, сильнее адаптация).

Примечание: официальные ноутбуки fine-tune'а Laya используют RLCD-цикл на K8s/`typed-decisions`-датасете (ноутбук Клэкера на 2x T4). Здесь более прямой путь под нашу задачу — чистая классификационная голова поверх того же энкодера, все шаги на одном GPU/CPU.

**План раздела:**
1. **4.1** — соберём текстовое представление state для train/test;
2. **4.2** — загрузим энкодер из чекпойнта Laya и повесим голову на `NUM_CLASSES`;
3. **4.3** — токенизация и DataLoader;
4. **4.4** — обучение (несколько эпох);
5. **4.5** — оценка на тесте и сводная таблица: zero-shot vs RF vs fine-tuned.


### 4.1 Текстовое представление сэмплов для обучения

In [ ]:
# 4.1 Строим компактный текст по каждому чанку (статистики + кривая) — входной код энкодера.
def state_to_text(chunk: np.ndarray, id_: int) -> str:
    s = chunk_to_state(chunk, id_)["stats"]
    parts = "; ".join(f"{k}={v:.3f}" for k, v in s.items())
    curve = ",".join(str(v) for v in np.round(chunk[::max(1, len(chunk)//12)], 1))
    return f"window stats: {parts} || power curve: {curve}"

X_train_text = [state_to_text(X_train[i], i) for i in range(len(X_train))]
X_test_text  = [state_to_text(X_test[i], i)  for i in range(len(X_test))]
print("Пример train-текста:", X_train_text[0][:160], "...")
print("Train:", len(X_train_text), "| Test:", len(X_test_text))

### 4.2 Модель: энкодер Laya + голова классификации

In [ ]:
# 4.2 Достаём энкодер Laya (тот же чекпойнт из кэша HF, что использует Router)
#     и ставим поверх классификационную голову с NUM_CLASSES выходов.
import torch
import torch.nn as nn

from laya import load as laya_load
agent_ft = laya_load("convaiinnovations/laya", device=DEVICE)
encoder_ft = agent_ft.model.encoder            # ModernBERT-large, hidden 1024
HIDDEN_DIM = encoder_ft.config.hidden_size
NUM_CLASSES = len(classes)                     # из ячейки 1.4
print("Энкодер:", encoder_ft.config.model_type, "| hidden:", HIDDEN_DIM, "| classes:", NUM_CLASSES)

class LayaClassifier(nn.Module):
    '''Энкодер Laya (заморожен или дообучается) + mean-pool + Linear -> NUM_CLASSES.'''
    def __init__(self, encoder, hidden, num_classes, freeze_encoder=True):
        super().__init__()
        self.encoder = encoder
        for p in self.encoder.parameters():
            p.requires_grad = not freeze_encoder
        self.drop = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden, num_classes)   # НОВАЯ голова с числом классов из данных

    def forward(self, input_ids, attention_mask):
        h = self.encoder(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        pooled = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)      # mean-pool
        return self.classifier(self.drop(pooled))

FREEZE_ENCODER = True    # True: обучаем только голову; False: дообучаем и энкодер (lR ниже)
model = LayaClassifier(encoder_ft, HIDDEN_DIM, NUM_CLASSES, freeze_encoder=FREEZE_ENCODER)
model.to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Обучаемых параметров:", trainable)

### 4.3 Токенизация и DataLoader

In [ ]:
# 4.3 Токенизируем train/test тексты и собираем батчи.
from torch.utils.data import DataLoader, TensorDataset

MAX_SEQ = 256
enc_tr = agent_ft.tok(X_train_text, padding=True, truncation=True, max_length=MAX_SEQ, return_tensors="pt")
enc_te = agent_ft.tok(X_test_text,  padding=True, truncation=True, max_length=MAX_SEQ, return_tensors="pt")

train_ds = TensorDataset(enc_tr["input_ids"], enc_tr["attention_mask"], torch.as_tensor(y_train))
test_ds  = TensorDataset(enc_te["input_ids"], enc_te["attention_mask"], torch.as_tensor(y_test))

BATCH_SIZE = 16 if DEVICE == "cuda" else 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
print(f"Train: {len(train_loader)} батчей | Test: {len(test_loader)} батчей")

### 4.4 Обучение головы (и опционально энкодера)

In [ ]:
# 4.4 Простой цикл обучения (несколько эпох, AdamW, кросс-энтропия).
import torch.nn.functional as F

EPOCHS = 3
LR = 3e-4 if FREEZE_ENCODER else 2e-5    # чтобы не сломать энкодер при дообучении
weight_decay = 0.01

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(params, lr=LR, weight_decay=weight_decay)

model.train()
for epoch in range(1, EPOCHS + 1):
    total_loss, correct, n = 0.0, 0, 0
    for input_ids, attention_mask, yb in train_loader:
        input_ids, attention_mask, yb = (
            input_ids.to(DEVICE), attention_mask.to(DEVICE), yb.to(DEVICE))
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(yb)
        correct += (logits.argmax(-1) == yb).sum().item()
        n += len(yb)
    print(f"epoch {epoch}/{EPOCHS}: loss={total_loss/n:.4f} train_acc={correct/n:.4f}")

# Сохранение весов головы (опционально)
# torch.save(model.state_dict(), "laya_head.pt")

### 4.5 Оценка и сравнение

In [ ]:
# 4.5 Оценка на том же тесте и сводная таблица по всем подходам.
model.eval()
preds_ft = []
with torch.no_grad():
    for input_ids, attention_mask, _ in test_loader:
        input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)
        preds_ft += model(input_ids, attention_mask).argmax(-1).tolist()

ft_acc = accuracy_score(y_test, preds_ft)
print(f"[Дообученная Laya] accuracy: {ft_acc:.4f}")
print()
print("=" * 46)
print(f"{'Подход':<28}{'accuracy':>10}")
print("-" * 46)
print(f"{'Laya zero-shot (2.4)':<28}{acc:>10.4f}")
print(f"{'RandomForest (3.1)':<28}{rf_acc:>10.4f}")
print(f"{'Laya fine-tuned (4.4)':<28}{ft_acc:>10.4f}")
print("=" * 46)

print()
print(classification_report(y_test, preds_ft))

## 5. Франкенштейн: энкодер Google TimesFM + классификатор Laya

Фокус-идея: **энкодер берём у TimesFM** (обучен на временных рядах → представление патчей `transformer_output`), **классификацию делает голова Laya** из 4.2. TimesFM отдаёт эмбеддинги размерности **1280** на патч, а голова Laya ждёт вектор **1024** — между ними вставляем **переходник (adapter)**: проекция + LayerNorm + ReLU.

Блоки:
1. **5.1** — загрузка `google/timesfm-3.0-pytorch` (backbone), тот же контекст 90 точек;
2. **5.2** — патчинг (уже знаем `input_patch_len=32`) + переходник `1280→1024` + голова Laya (`NUM_CLASSES`);
3. **5.3** — токенизация заменяется на тензоры рядов; DataLoader;
4. **5.4** — обучение (та же схема, что в 4.4);
5. **5.5** — оценка на том же тесте и финальная сравнительная таблица.

Трансформер TimesFM замораживаем, обучаем только переходник и голову Laya.

### 5.1 Загрузка Google TimesFM 3.0 как энкодера

In [ ]:
# 5.1 Установка и загрузка TimesFM 3.0 (backbone для энкодера).
#     API тот же, что в исходном ноутбуке TimesFM3_DeepSeek.ipynb.
%pip install -q timesfm
from timesfm3 import TimesFM3Forecaster

forecaster = TimesFM3Forecaster.from_pretrained(
    "google/timesfm-3.0-pytorch",
    per_core_batch_size=4,
)
print(f"TimesFM загружен на устройстве: {forecaster.device}")

backbone_tf = forecaster.model            # внутренняя PyTorch-модель (backbone)
HIDDEN_TF = 1280                          # размерность эмбеддинга патча у TimesFM 3.0
INPUT_PATCH = backbone_tf.input_patch_len # 32
print("input_patch_len:", INPUT_PATCH, "| hidden_dim:", HIDDEN_TF)

### 5.2 Переходник берёт «скрытые состояния» (как в Part 3 исходника)

In [ ]:
# 5.2 Патчим ряд (аналог ячейки Part 3 исходника) и вытаскиваем transformer_output,
#     затем переходник 1280 -> 1024 и голову Laya.
import torch.nn as nn

def pad_to_patches(x: torch.Tensor):
    '''x: (B, L) -> values (B, 1, n_patches, p) + masks + patch_is_target.'''
    batch_size, seq_len = x.shape
    n_patches = (seq_len + INPUT_PATCH - 1) // INPUT_PATCH
    padded_len = n_patches * INPUT_PATCH
    pad = padded_len - seq_len
    x_padded = torch.nn.functional.pad(x, (0, pad)) if pad > 0 else x
    mask = torch.zeros(batch_size, 1, padded_len, dtype=torch.bool, device=x.device)
    mask[:, :, seq_len:] = True
    values = x_padded.view(batch_size, 1, n_patches, INPUT_PATCH)
    masks = mask.view(batch_size, 1, n_patches, INPUT_PATCH)
    patch_is_target = torch.ones(batch_size, 1, n_patches, dtype=torch.bool, device=x.device)
    return {'values': values, 'masks': masks, 'patch_is_target': patch_is_target}

def timesfm_embed(backbone, x: torch.Tensor) -> torch.Tensor:
    '''Прямой проход backbone с запросом скрытых состояний: (B, n_patches, HIDDEN_TF).'''
    inputs = pad_to_patches(x)
    out = backbone.forward(inputs, return_aux_outputs=True)
    return out['__call__:transformer_output'].squeeze(1)

class TimesFM2Laya(nn.Module):
    '''Переходник: патчи TimesFM -> mean-pool -> проекция 1280->1024 -> голова Laya.'''
    def __init__(self, backbone, hidden_tf, hidden_laya, num_classes, freeze_encoder=True):
        super().__init__()
        self.backbone = backbone
        if freeze_encoder:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.adapter = nn.Sequential(                    # ПЕРЕХОДНИК 1280 -> 1024
            nn.Linear(hidden_tf, hidden_laya),
            nn.LayerNorm(hidden_laya),
            nn.ReLU(),
        )
        self.drop = nn.Dropout(0.1)
        self.classifier = nn.Linear(hidden_laya, num_classes)   # голова Laya (как в 4.2)

    def forward(self, x):
        patches = timesfm_embed(self.backbone, x)        # (B, n_patches, 1280)
        pooled = patches.mean(dim=1)                     # mean-pool по патчам -> (B, 1280)
        h = self.adapter(pooled)                         # -> (B, 1024)
        return self.classifier(self.drop(h))

HIDDEN_LAYA = 1024
FREEZE_ENCODER = True    # True: обучаем переходник + голову; False: и TimesFM (LR ниже)
franken_model = TimesFM2Laya(backbone_tf, HIDDEN_TF, HIDDEN_LAYA, NUM_CLASSES,
                             freeze_encoder=FREEZE_ENCODER).to(DEVICE)
trainable = sum(p.numel() for p in franken_model.parameters() if p.requires_grad)
print("Обучаемых параметров:", trainable)

### 5.3 Датасет: raw-ряды (патчи) вместо текста

In [ ]:
# 5.3 Собираем батчи из сырых чанков (90 точек). Тексты/токены для TimesFM не нужны.
from torch.utils.data import TensorDataset, DataLoader

Xtr_t = torch.tensor(X_train, dtype=torch.float32)
Xte_t = torch.tensor(X_test,  dtype=torch.float32)
ytr_t = torch.tensor(y_train)
yte_t = torch.tensor(y_test)

train_ds = TensorDataset(Xtr_t, ytr_t)
test_ds  = TensorDataset(Xte_t, yte_t)

BATCH_SIZE = 16 if DEVICE == "cuda" else 8
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False)
print("Форма батча:", next(iter(train_loader))[0].shape)

### 5.4 Обучение переходника + головы Laya

In [ ]:
# 5.4 Обучение. Та же схема, что в 4.4: CE, AdamW, freeze backbone.
import torch.nn.functional as F

EPOCHS = 3
LR = 3e-4 if FREEZE_ENCODER else 2e-5
opt = torch.optim.AdamW([p for p in franken_model.parameters() if p.requires_grad],
                        lr=LR, weight_decay=0.01)

franken_model.train()
for epoch in range(1, EPOCHS + 1):
    tot, corr, n = 0.0, 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = franken_model(xb)
        loss = F.cross_entropy(logits, yb)
        loss.backward()
        opt.step()
        tot += loss.item() * len(yb)
        corr += (logits.argmax(-1) == yb).sum().item()
        n += len(yb)
    print(f"epoch {epoch}/{EPOCHS}: loss={tot/n:.4f} train_acc={corr/n:.4f}")

### 5.5 Оценка и финальная сводка (все подходы, один тест)

In [ ]:
# 5.5 Оценка Frankensteini на тесте и финальная сравнительная таблица.
franken_model.eval()
preds_fr = []
with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(DEVICE)
        preds_fr += franken_model(xb).argmax(-1).tolist()

fr_acc = accuracy_score(y_test, preds_fr)
print(f"[Frankenstein TimesFM+Laya] accuracy: {fr_acc:.4f}")
print()
print("=" * 52)
print(f"{'Подход':<32}{'accuracy':>10}")
print("-" * 52)
print(f"{'Laya zero-shot (2.4)':<32}{acc:>10.4f}")
print(f"{'RandomForest stats (3.1)':<32}{rf_acc:>10.4f}")
print(f"{'Laya fine-tuned (4.4)':<32}{ft_acc:>10.4f}")
print(f"{'TimesFM+Laya adapter (5.4)':<32}{fr_acc:>10.4f}")
print("=" * 52)

### 5.6 Linear-probe: «потолок» признаков TimesFM (диагностика)

**Что это:** на **замороженных** эмбеддингах TimesFM обучаем простой классификатор (LogReg / RF) без переходников.
- Если «потолок» ≈ случайному — признаки TimesFM плохо отделяют классы, и улучшать нечего;
- если «потолок» высокий, а наша голова 5.4 ниже — значит, дело в **адаптере/обучении**, и 5.7 имеет смысл.

Здесь же исправляем два найденных дефекта входа: **z-нормализация** ряда (TimesFM обучен на нормированных данных) и **маскированный mean-pool** — усреднение только по реальным точкам патчей (в 5.2–5.4 усреднялись в т.ч. padding-патчи).

In [ ]:
# 5.6 Linear-probe: признаки TimesFM без переходника.
import torch
import torch.nn as nn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader, TensorDataset

class TimesFMEncoder(nn.Module):
    '''Маскированный кодировщик: z-норм + патчи + masked mean-pool -> 1280.'''
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        for p in self.backbone.parameters():
            p.requires_grad = False

    def forward(self, x):
        x = (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)  # z-норм
        inputs = pad_to_patches(x)
        out = self.backbone.forward(inputs, return_aux_outputs=True)
        patches = out['__call__:transformer_output'].squeeze(1)   # (B, n_patches, D)
        real = (~inputs['masks'].squeeze(1)).float()              # (B, n_patches, P)
        w = real.sum(-1)                                          # (B, n_patches)  сколько реальных точек в патче
        w = w / w.sum(-1, keepdim=True).clamp(min=1e-9)           # веса патчей (нормированы)
        return (patches * w.unsqueeze(-1)).sum(1)                 # masked mean-pool (B, D)

tf_enc = TimesFMEncoder(backbone_tf).to(DEVICE)
print("Энкодер TimesFM-Lite готов. HIDDEN:", HIDDEN_TF)

def embed_tf(model, X, device, batch=64):
    model.eval()
    outs = []
    for xb in DataLoader(torch.tensor(X, dtype=torch.float32), batch_size=batch):
        outs.append(model(xb.to(device)).detach().cpu().numpy())
    return np.vstack(outs)

print("Извлекаем эмбеддинги (может занять время)...")
Etr = embed_tf(tf_enc, X_train, DEVICE)
Ete = embed_tf(tf_enc, X_test,  DEVICE)
print("Train:", Etr.shape, "| Test:", Ete.shape)

sc_probe = StandardScaler().fit(Etr)
Etr_s, Ete_s = sc_probe.transform(Etr), sc_probe.transform(Ete)

for name, clf in [
    ("LogisticRegression", LogisticRegression(max_iter=2000, C=1.0)),
    ("RandomForest",       RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
]:
    clf.fit(Etr_s, y_train)
    p = clf.predict(Ete_s)
    print(f"[Probe {name}]  acc={accuracy_score(y_test, p):.4f}  "
          f"macro-F1={f1_score(y_test, p, average='macro'):.4f}")

### 5.7 Улучшенный Франкенштейн: bottleneck-адаптер + masked-pool + z-норм

In [ ]:
# 5.7 V2: применяем уроки критразбора.
#     1) z-нормализация входа и заморозка TimesFM (как в 5.4);
#     2) МАСКИРОВАННЫЙ mean-pool (не трогаем padding);
#     3) bottleneck 1280 -> 256 -> 1024 (меньше свободных параметров, чем 1280->1024);
#     4) та же голова Laya Linear(1024, NUM_CLASSES), классификатор переносим из 4.4 при наличии.

class TimesFM2LayaBottleneck(nn.Module):
    def __init__(self, backbone, hidden_tf, hidden_laya, num_classes, freeze=True):
        super().__init__()
        self.encoder_tf = TimesFMEncoder(backbone)      # z-норм + маскированный pool
        if freeze:
            for p in self.encoder_tf.parameters():
                p.requires_grad = False
        self.adapter = nn.Sequential(                   # bottleneck 1280 -> 256 -> 1024
            nn.Linear(hidden_tf, 256), nn.ReLU(),
            nn.Linear(256, hidden_laya),
        )
        self.drop = nn.Dropout(0.3)
        self.classifier = nn.Linear(hidden_laya, num_classes)   # голова Laya (как 4.2)

    def forward(self, x):
        h = self.encoder_tf(x)                          # (B, 1280)
        h = self.adapter(h)                             # (B, 1024)
        return self.classifier(self.drop(h))

FREEZE_ENCODER = True
frank2 = TimesFM2LayaBottleneck(backbone_tf, HIDDEN_TF, HIDDEN_LAYA, NUM_CLASSES,
                                freeze=FREEZE_ENCODER).to(DEVICE)

# Переносим веса головы из 4.4 (если она выполнена), иначе обучаем с нуля
try:
    head_src = model.state_dict()
    head_sd  = {k.removeprefix('classifier.'): v for k, v in head_src.items()
                if k.startswith('classifier.')}
    frank2.classifier.load_state_dict(head_sd, strict=False)
    print("Голова из 4.4 перенесена в 5.7.")
except Exception as e:
    print("(голову из 4.4 не перенесли:", e, ")")

trainable = sum(p.numel() for p in frank2.parameters() if p.requires_grad)
print("Обучаемых параметров:", trainable)

### 5.7.1 Обучение V2 с валидационным под-сплитом (train остаётся train, тест не трогаем)

In [ ]:
# 5.7.1 Обучение улучшенной модели. Валид-сплит берём ИЗ train — тестовый сплит (1.4) не трогаем,
#      чтобы честно сравнивать со всеми подходами выше.
from sklearn.utils.class_weight import compute_class_weight

Xv_tr, Xv_va, yv_tr, yv_va = train_test_split(
    X_train, y_train, test_size=0.15, random_state=42, stratify=y_train)
print("Val-сплит:", Xv_tr.shape, Xv_va.shape)

cw = compute_class_weight('balanced', classes=np.unique(yv_tr), y=yv_tr)
crit = nn.CrossEntropyLoss(weight=torch.tensor(cw, dtype=torch.float32).to(DEVICE))
opt = torch.optim.AdamW(
    [p for p in frank2.parameters() if p.requires_grad], lr=3e-4, weight_decay=0.01)

BATCH = 16 if DEVICE == "cuda" else 8
loader_tr = DataLoader(TensorDataset(torch.tensor(Xv_tr, dtype=torch.float32),
                                     torch.tensor(yv_tr)), batch_size=BATCH, shuffle=True)
loader_va = DataLoader(TensorDataset(torch.tensor(Xv_va, dtype=torch.float32),
                                     torch.tensor(yv_va)), batch_size=BATCH)

EPOCHS = 5
best_va, best_state = 0.0, None
for ep in range(1, EPOCHS + 1):
    frank2.train()
    tot_l, tot_c, tot_n = 0.0, 0, 0
    for xb, yb in loader_tr:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        opt.zero_grad()
        logits = frank2(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        tot_l += loss.item() * len(yb)
        tot_c += (logits.argmax(-1) == yb).sum().item()
        tot_n += len(yb)
    frank2.eval()
    va_preds = []
    with torch.no_grad():
        for xb, _ in loader_va:
            va_preds += frank2(xb.to(DEVICE)).argmax(-1).tolist()
    va_acc = accuracy_score(yv_va, va_preds)
    print(f"epoch {ep}/{EPOCHS}: loss={tot_l/tot_n:.4f} train_acc={tot_c/tot_n:.4f} val_acc={va_acc:.4f}")
    if va_acc > best_va:
        best_va = va_acc
        best_state = {k: v.clone() for k, v in frank2.state_dict().items()}

if best_state is not None:
    frank2.load_state_dict(best_state)
    print("Загружена лучшая эпоха по val_acc:", best_va)

### 5.7.2 Оценка V2 на тесте и окончательная сводная таблица

In [ ]:
# 5.7.2 Финальная оценка улучшенного Франкенштейна на том же тесте.
from sklearn.metrics import f1_score

frank2.eval()
preds_fr2 = []
with torch.no_grad():
    for xb in DataLoader(torch.tensor(X_test, dtype=torch.float32), batch_size=BATCH):
        preds_fr2 += frank2(xb.to(DEVICE)).argmax(-1).tolist()

fr2_acc = accuracy_score(y_test, preds_fr2)
fr2_f1  = f1_score(y_test, preds_fr2, average="macro")
print(f"[Frankenstein V2] acc={fr2_acc:.4f}  macro-F1={fr2_f1:.4f}")

print()
print("=" * 52)
print(f"{'Подход':<38}{'acc':>7}{'F1_macro':>9}")
print("-" * 52)
print(f"{'Laya zero-shot (2.4)':<38}{acc:>7.4f}{f1_score(y_true_sub, preds, average='macro'):>9.4f}")
print(f"{'RandomForest stats (3.1)':<38}{rf_acc:>7.4f}{'  --':>9}")
try:
    print(f"{'Laya fine-tuned (4.4)':<38}{ft_acc:>7.4f}{f1_score(y_test, preds_ft, average='macro'):>9.4f}")
except NameError:
    print(f"{'Laya fine-tuned (4.4)':<38}(пропущен раздел 4){'':>9}")
print(f"{'Franken V1 (5.4)':<38}{fr_acc:>7.4f}{f1_score(y_test, preds_fr, average='macro'):>9.4f}")
print(f"{'Franken V2 (5.7)':<38}{fr2_acc:>7.4f}{fr2_f1:>9.4f}")
print("=" * 52)

## 6. Выводы

- Сравните итоговую таблицу 5.7.2: все подходы оценены на **одном** тесте (30% стратифицированного сплита из 1.4).
- Если сэмплов меньше ~1000 и собственных классов немного — начинайте с `FREEZE_ENCODER = True` (надежно, дёшево); при данных больше и желании сильнее адаптировать представления — `FREEZE_ENCODER = False`.
- 4.1 даёт только статистики + упрощённую кривую. Если точность головы упёрлась — обогащайте текст признаками/эмбеддингами TimesFM из исходного ноутбука (аналог того, что делал Клэкер в fine-tune Laya).
- В защищённом контуре веса дообученной головы сохраняются в `laya_head.pt` и применяются вне сети без повторной генерации.
